[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HSF-reco-and-software-triggers/Tracking-ML-Exa.TrkX/blob/master/Examples/TrackML_Quickstart/DM_colab_quickstart.ipynb)

# TrackML Quickstart

## Install Libraries

**Note: Before running notebook, ensure your runtime is set to GPU**

First, we just install a few libraries (this should take around 5 minutes and automatically restart the kernel), and load in the repository.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!pip install seaborn bokeh 
!conda install pandas scipy matplotlib cupy "cudatoolkit>=11.3" "pytorch>=1.10.2" "pytorch-lightning>=1.6" pyg faiss-gpu -c pytorch -c pyg -c conda-forge

In [ ]:
!git clone https://github.com/HSF-reco-and-software-triggers/Tracking-ML-Exa.TrkX.git
%cd Tracking-ML-Exa.TrkX/Examples/TrackML_Quickstart

# Import libraries

In [1]:
import sys, os
sys.path.append("../../")
from Scripts import train_metric_learning, run_metric_learning_inference, train_gnn, run_gnn_inference, build_track_candidates, evaluate_candidates
from Scripts.utils.convenience_utils import get_example_data, plot_true_graph, get_training_metrics, plot_training_metrics, plot_neighbor_performance, plot_predicted_graph, plot_track_lengths, plot_edge_performance, plot_graph_sizes
import yaml

import warnings
warnings.filterwarnings("ignore")
CONFIG = 'pipeline_config_quirk.yaml'
# pipeline_config_quirk.yaml

Loading BokehJS ...

/home/waqar/miniconda3/envs/trackml-quickstart/lib/python3.12/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/home/waqar/miniconda3/envs/trackml-quickstart/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading BokehJS ...

## Download Data

In [ ]:
%%capture
!mkdir datasets
!wget https://portal.nersc.gov/cfs/m3443/dtmurnane/TrackML_Example/trackml_quickstart_dataset.tar.gz -O datasets/trackml_quickstart_dataset.tar.gz

In [ ]:
%%capture
!tar -xvf datasets/trackml_quickstart_dataset.tar.gz -C datasets;
!rm datasets/trackml_quickstart_dataset.tar.gz

## TrackML Dataset

The TrackML dataset contains simulated indepedent proton-proton collision events, each generating hundreds of particles, each of which hits cells and layers of the detector layers multiple times. The detector records the spatial coordinates and other auxillary information of these hits which, if properly connected, form tracks associated with the parent particle and the collision event from which it originates. The challenge and goal of this project is to associate each and every hit to one single track with optimal purity and efficiency, whose precise definition will be given later.

Each entry in the particles data frame contains a unique identifier of the particle (particle_id), its charge (q), its initial position or vertex $(v_x, v_y, v_z)$, its initial momentum in GeV/c $(p_x, p_y, p_z)$ and its associated number of detector hits. 

Many particles do not leave behind any detector hits and obviously cannot be associated to any track. This is called "detector inefficiency". They are among "uninterested particles" and will be mostly filtered out by a simple momentum cut.

### Training data
Let us take a look at the data before training. In this example pipeline, we have preprocessed the TrackML data into a more convenient form. We calculated directional information and summary statistics from the charge deposited in each spacepoints, and append them to its cyclidrical coordinates. Let us load an example data file and inspect the content.

In [2]:
with open(CONFIG, 'r') as f:
    configs = yaml.load(f, Loader=yaml.FullLoader)

In [3]:
example_data_df, example_data_pyg = get_example_data(configs)
example_data_df.head()

,0,1,2
0,0.031687,0.567258,0.012397
1,0.071077,0.566757,0.028057
2,0.072884,0.566715,0.028778
3,0.170848,0.565448,0.067714
4,0.172741,0.565432,0.068462


### Visualize tracks

A "true track" is defined as a set of sequential hits, all left by the same particle. Therefore a true edge is the edge formed by two sequential hits. Let's visualize a random set of 200 true tracks:

In [4]:
plot_true_graph_fig = plot_true_graph(example_data_pyg, num_tracks=200)

# 1. Train Metric Learning

## Train metric learning model

Finally we come to model training. By default, we train the MLP for 30 epochs, which takes approximately 15 minutes on an NVidia V100. Feel free to adjust the epoch number in pipeline_config.yml

In [5]:
metric_learning_trainer, metric_learning_model = train_metric_learning(CONFIG)

INFO:-------------------- Step 1: Running metric learning training --------------------
INFO:----------------------------- a) Initialising model -----------------------------
INFO:------------------------------ b) Running training ------------------------------
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
Missing logger folder: artifacts/metric_learning/trackml_quickstart_quirk
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params
---------------------------------------
0 | network | Sequential | 3.2 

Epoch 49: 100%|██████████| 90/90 [00:14<00:00,  6.20it/s, loss=0.00732, v_num=0]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 90/90 [00:14<00:00,  6.16it/s, loss=0.00732, v_num=0]

INFO:-------------------------------- c) Saving model --------------------------------


## Plot training metrics

We can examine how the training went. This is stored in a simple dataframe:

In [6]:
embedding_metrics = get_training_metrics(metric_learning_trainer)
embedding_metrics.head()

,epoch,train_loss,val_loss,eff,pur,current_lr
0,0,0.008525,0.005163,0.875167,0.100068,0.000125
1,1,0.008316,0.004860,0.874860,0.152399,0.000250
2,2,0.008249,0.004877,0.872044,0.156414,0.000375
3,3,0.008143,0.004791,0.875375,0.166659,0.000350
4,4,0.008147,0.004831,0.873058,0.167607,0.000625


In [7]:
embedding_training_figs = plot_training_metrics(embedding_metrics)

## Evaluate model performance on sample test data

Here we evaluate the model performace on one sample test data. We look at how the efficiency and purity change with the embedding radius.

In [8]:
neighbor_perf_figs = plot_neighbor_performance(metric_learning_model)

## Plot example truth and predicted graphs

In [9]:
predicted_graph_figs = plot_predicted_graph(metric_learning_model)

## Track lengths

In [10]:
track_length_figs = plot_track_lengths(metric_learning_model)

In [11]:
graph_sizes_fig = plot_graph_sizes(metric_learning_model)

100%|██████████| 80/80 [00:12<00:00,  6.64it/s]


# 2. Construct graphs from metric learning inference

This step performs model inference on the entire input datasets (train, validation and test), to obtain input graphs to the graph neural network. Optionally, we also clear the directory.

In [12]:
graph_builder = run_metric_learning_inference(CONFIG)

INFO:------------- Step 2: Constructing graphs from metric learning model -------------
INFO:---------------------------- a) Loading trained model ----------------------------
INFO:----------------------------- b) Running inferencing -----------------------------


Training finished, running inference to build graphs...


100%|██████████| 10/10 [00:01<00:00,  7.05it/s]


In [13]:
import os, gc, torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"Cleared CUDA cache on {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available")

print("Run this cell right before Step 3. If allocator behavior is unchanged, restart kernel once.")


Cleared CUDA cache on NVIDIA GeForce RTX 4090
Run this cell right before Step 3. If allocator behavior is unchanged, restart kernel once.


# 3. Train graph neural networks

We have a set of graphs constructed. We now train a GNN to classify edges as either "true" (belonging to the same track) or "false" (not belonging to the same track).

In [14]:
gnn_trainer, gnn_model = train_gnn(CONFIG)

INFO:-------------------------  Step 3: Running GNN training  -------------------------
INFO:----------------------------- a) Initialising model -----------------------------
INFO:------------------------------ b) Running training ------------------------------
Using 16bit None Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: artifacts/gnn/trackml_quickstart_quirk
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                   | Type       | Params
------------------------------------------------------
0 | node_encoder           | Sequential | 9.9 K 
1 | edge_encoder           | Sequential | 28.0 K
2 | edge_network           | Sequential | 37.2 K
3 | node_network           | Sequential | 37.2 K
4 | output_edge_classifier | Sequential | 37.5 K
------------------------------------------------------
149 K     Trainable params
0

Epoch 29: 100%|██████████| 90/90 [00:03<00:00, 26.92it/s, loss=0.132, v_num=0]

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 90/90 [00:03<00:00, 26.78it/s, loss=0.132, v_num=0]

INFO:-------------------------------- c) Saving model --------------------------------


## Plot training metrics

In [15]:
gnn_metrics = get_training_metrics(gnn_trainer)
gnn_metrics.head()

,epoch,train_loss,val_loss,eff,pur,current_lr
0,0,0.559687,0.599877,0.981174,0.755144,0.0002
1,1,0.538885,0.546296,0.966507,0.777798,0.0004
2,2,0.534299,0.541574,0.986725,0.752876,0.0006
3,3,0.528948,0.546423,0.990593,0.747073,0.0008
4,4,0.519132,0.537015,0.981786,0.763494,0.0010


In [16]:
gnn_training_figs = plot_training_metrics(gnn_metrics)

## Evaluate model performance on sample test data

Here we evaluate the model performace on one sample test data. We look at how the efficiency and purity change with the embedding radius.

In [17]:
edge_perf_figs = plot_edge_performance(gnn_model)

# Step 4: GNN inference 

In [18]:
run_gnn_inference(CONFIG)

INFO:--------------------- Step 4: Scoring graph edges using GNN  ---------------------
INFO:---------------------------- a) Loading trained model ----------------------------
INFO:----------------------------- b) Running inferencing -----------------------------


Training finished, running inference to filter graphs...
Building train


100%|██████████| 80/80 [00:00<00:00, 196.60it/s]


Building val


100%|██████████| 10/10 [00:00<00:00, 191.72it/s]


Building test


100%|██████████| 10/10 [00:00<00:00, 143.59it/s]


# Step 5: Build track candidates from GNN

In [19]:
build_track_candidates(CONFIG)

INFO:-----------  Step 5: Building track candidates from the scored graph  -----------
INFO:---------------------------- a) Loading scored graphs ----------------------------
INFO:---------------------------- b) Labelling graph nodes ----------------------------
100%|██████████| 100/100 [00:00<00:00, 828.46it/s]


# Step 6: Evaluate track candidates

We can control the matching style in the pipeline config file. The following all require at least a majority of hits to match in each scheme (i.e. matching fraction = 50%).
A discussion of each matching style and some worked examples can be found in the [Documentation](https://hsf-reco-and-software-triggers.github.io/Tracking-ML-Exa.TrkX/performance/matching_definitions/).

ATLAS style matching is the default.

In [20]:
evaluated_events, reconstructed_particles, particles, matched_tracks, tracks = evaluate_candidates(CONFIG)

INFO:------------ Step 6: Evaluating the track reconstruction performance ------------
INFO:--------------------------- a) Loading labelled graphs ---------------------------
100%|██████████| 100/100 [00:00<00:00, 174.72it/s]
INFO:--------------------- b) Calculating the performance metrics ---------------------
INFO:Number of reconstructed particles: 25088
INFO:Number of particles: 26506
INFO:Number of matched tracks: 31052
INFO:Number of tracks: 31248
INFO:Number of duplicate reconstructed particles: 5956
INFO:Efficiency: 0.947
INFO:Fake rate: 0.006
INFO:Duplication rate: 0.237
INFO:------------------------------ c) Plotting results ------------------------------


In [21]:
# Optional: save selected plots from this notebook
import os
from pathlib import Path

import matplotlib.pyplot as plt
from bokeh.io import export_png

SAVE_PLOTS = True  # Set True when you want to export
OUTPUT_DIR = Path("saved_plots")
MATPLOTLIB_DPI = 500
BOKEH_SCALE_FACTOR = 5

# Choose what to save
TO_SAVE = [
    "plot_true_graph",
    "embedding_training",
    "neighbor_performance",
    "predicted_graph",
    "track_lengths",
    "graph_sizes",
    "gnn_training",
    "edge_performance",
]

plot_registry = {
    "plot_true_graph": locals().get("plot_true_graph_fig"),
    "embedding_training": locals().get("embedding_training_figs"),
    "neighbor_performance": locals().get("neighbor_perf_figs"),
    "predicted_graph": locals().get("predicted_graph_figs"),
    "track_lengths": locals().get("track_length_figs"),
    "graph_sizes": locals().get("graph_sizes_fig"),
    "gnn_training": locals().get("gnn_training_figs"),
    "edge_performance": locals().get("edge_perf_figs"),
}

def _bokeh_figures(obj):
    if obj is None:
        return []
    if isinstance(obj, dict) and "figures" in obj:
        return obj["figures"]
    return [obj]

def _save_matplotlib(fig, outpath):
    fig.savefig(outpath, dpi=MATPLOTLIB_DPI, bbox_inches="tight")

def _save_bokeh(fig, outpath):
    # Requires selenium + browser driver for PNG export.
    export_png(fig, filename=str(outpath), scale_factor=BOKEH_SCALE_FACTOR)

if SAVE_PLOTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    saved, skipped, failed = [], [], []

    for name in TO_SAVE:
        obj = plot_registry.get(name)
        if obj is None:
            skipped.append((name, "plot object not found (run the source plotting cell first)"))
            continue

        if name == "graph_sizes":
            try:
                out = OUTPUT_DIR / f"{name}.png"
                _save_matplotlib(obj, out)
                saved.append(str(out))
            except Exception as exc:
                failed.append((name, str(exc)))
            continue

        figs = _bokeh_figures(obj)
        for idx, fig in enumerate(figs, start=1):
            suffix = f"_{idx}" if len(figs) > 1 else ""
            out = OUTPUT_DIR / f"{name}{suffix}.png"
            try:
                _save_bokeh(fig, out)
                saved.append(str(out))
            except Exception as exc:
                failed.append((f"{name}{suffix}", str(exc)))

    print(f"Saved {len(saved)} plot files")
    if saved:
        print("\n".join(saved))
    if skipped:
        print("\nSkipped:")
        for n, msg in skipped:
            print(f"- {n}: {msg}")
    if failed:
        print("\nFailed:")
        for n, msg in failed:
            print(f"- {n}: {msg}")
else:
    print("Set SAVE_PLOTS = True and rerun this cell to export selected plots.")

Saved 1 plot files
saved_plots/graph_sizes.png

Failed:
- plot_true_graph: Neither firefox and geckodriver nor a variant of chromium browser and chromedriver are available on system PATH. You can install the former with 'conda install -c conda-forge firefox geckodriver'.
- embedding_training_1: Neither firefox and geckodriver nor a variant of chromium browser and chromedriver are available on system PATH. You can install the former with 'conda install -c conda-forge firefox geckodriver'.
- embedding_training_2: Neither firefox and geckodriver nor a variant of chromium browser and chromedriver are available on system PATH. You can install the former with 'conda install -c conda-forge firefox geckodriver'.
- embedding_training_3: Neither firefox and geckodriver nor a variant of chromium browser and chromedriver are available on system PATH. You can install the former with 'conda install -c conda-forge firefox geckodriver'.
- neighbor_performance_1: Neither firefox and geckodriver nor a v

In [22]:
print(plot_true_graph_fig if "plot_true_graph_fig" in locals() else "missing")
print(embedding_training_figs if "embedding_training_figs" in locals() else "missing")


figure(id='p1008', ...)
{'layout': Row(id='p3366', ...), 'figures': [figure(id='p3179', ...), figure(id='p3256', ...), figure(id='p3311', ...)]}


In [23]:
# Save notebook plots: Bokeh as individual HTML files, Matplotlib as PNG
from pathlib import Path

from bokeh.io import output_file, save
import matplotlib.pyplot as plt

SAVE_PLOTS = True
OUTPUT_DIR = Path("saved_plots")
MATPLOTLIB_DPI = 600

TO_SAVE = [
    "plot_true_graph",
    "embedding_training",
    "neighbor_performance",
    "predicted_graph",
    "track_lengths",
    "graph_sizes",
    "gnn_training",
    "edge_performance",
]

plot_registry = {
    "plot_true_graph": locals().get("plot_true_graph_fig"),
    "embedding_training": locals().get("embedding_training_figs"),
    "neighbor_performance": locals().get("neighbor_perf_figs"),
    "predicted_graph": locals().get("predicted_graph_figs"),
    "track_lengths": locals().get("track_length_figs"),
    "graph_sizes": locals().get("graph_sizes_fig"),
    "gnn_training": locals().get("gnn_training_figs"),
    "edge_performance": locals().get("edge_perf_figs"),
}

def _bokeh_figures(obj):
    if obj is None:
        return []
    if isinstance(obj, dict) and "figures" in obj:
        return obj["figures"]
    return [obj]

def _save_matplotlib(fig, outpath):
    fig.savefig(outpath, dpi=MATPLOTLIB_DPI, bbox_inches="tight")

def _save_bokeh_html(fig, outpath):
    output_file(str(outpath), title=outpath.stem)
    save(fig)

if SAVE_PLOTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    saved, skipped, failed = [], [], []

    for name in TO_SAVE:
        obj = plot_registry.get(name)
        if obj is None:
            skipped.append((name, "plot object not found (run the source plotting cell first)"))
            continue

        if name == "graph_sizes":
            try:
                out = OUTPUT_DIR / f"{name}.png"
                _save_matplotlib(obj, out)
                saved.append(str(out))
            except Exception as exc:
                failed.append((name, str(exc)))
            continue

        figs = _bokeh_figures(obj)
        for idx, fig in enumerate(figs, start=1):
            suffix = f"_{idx}" if len(figs) > 1 else ""
            out = OUTPUT_DIR / f"{name}{suffix}.html"
            try:
                _save_bokeh_html(fig, out)
                saved.append(str(out))
            except Exception as exc:
                failed.append((f"{name}{suffix}", str(exc)))

    print(f"Saved {len(saved)} plot files")
    if saved:
        print("\n".join(saved))
    if skipped:
        print("\nSkipped:")
        for n, msg in skipped:
            print(f"- {n}: {msg}")
    if failed:
        print("\nFailed:")
        for n, msg in failed:
            print(f"- {n}: {msg}")
else:
    print("Set SAVE_PLOTS = True and rerun this cell to export selected plots.")


INFO:bokeh.io.state:Session output file 'saved_plots/plot_true_graph.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/embedding_training_1.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/embedding_training_2.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/embedding_training_3.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/neighbor_performance_1.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/neighbor_performance_2.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/neighbor_performance_3.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/predicted_graph_1.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/predicted_gra

Saved 17 plot files
saved_plots/plot_true_graph.html
saved_plots/embedding_training_1.html
saved_plots/embedding_training_2.html
saved_plots/embedding_training_3.html
saved_plots/neighbor_performance_1.html
saved_plots/neighbor_performance_2.html
saved_plots/neighbor_performance_3.html
saved_plots/predicted_graph_1.html
saved_plots/predicted_graph_2.html
saved_plots/track_lengths_1.html
saved_plots/track_lengths_2.html
saved_plots/graph_sizes.png
saved_plots/gnn_training_1.html
saved_plots/gnn_training_2.html
saved_plots/gnn_training_3.html
saved_plots/edge_performance_1.html
saved_plots/edge_performance_2.html
